In [ ]:
#CREACION DE CARPETAS
!mkdir -p datos/crudos
!mkdir -p datos/procesados
!mkdir -p datos/externos
print("Carpetas creadas !!!!!!")

In [ ]:
%pip install pyspark
print("SPARK INSTALADO")

In [ ]:
#IMPORTACION DE LIBRERIAS
import pandas as pd #Manejar datos como excel
import sqlite3 #Para base de datos ligera
import json #Abrir y manipular archivos json
from pathlib import Path #Atajos
from datetime import time #Con
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
#Rutas de trabajo ya definidas "atajos"
BASE_DIR = Path("/content/")
DATA_RAW = BASE_DIR/"datos"/"crudos"
DATA_EXTERNAL = BASE_DIR/"datos"/"externos"
DATA_PROCESSED = BASE_DIR/"datos"/"procesados"
print(f"Datos crudos-> {DATA_RAW}")
print(f"Datos procesados-> {DATA_PROCESSED}")
print(f"Datos externos -> {DATA_EXTERNAL}")
DOCUMENTOS = BASE_DIR/"datos"/"documentos"
DOCUMENTOS.mkdir(parents=True, exist_ok=True)
print(f"Carpetas creadas con exito en {DOCUMENTOS}")

In [ ]:
#CREANDO SECION DE SPARK
spark = SparkSession.builder\
 .master("local[*]")\
 .appName("avance_proyecto_final")\
 .getOrCreate()
print(f"Spark {spark.version}")
print(f"Paralelismo {spark.sparkContext.defaultParallelism}")

In [ ]:
#CARGAR DATASET LIMPIO DE LA GUIA DOS
ruta_descarga = DATA_PROCESSED/"dataset_limpio.csv"
print(f"LEYENDO ARCHIVO: {ruta_descarga}")

df = spark.read \
  .option("header", True)\
  .option("inferSchema", True)\
  .csv(str(ruta_descarga))

print("DATASET CARGADO EN SPARK")
print(f"FILA {df.count()}")
print(f"COLUMNAS {len(df.columns)}")

In [ ]:
#ver esquema
#EN PANDAS .info()
print("\n ESQUEMA DE DATOS: ")
df.printSchema()
print("\n PRIMERAS FILAS======")
df.show(5, truncate=False)

In [ ]:
#PARTICIONES
print(f"\n PARTICIONES INICIALES: {df.rdd.getNumPartitions()}")

In [ ]:
#FILTRAR REGISTROS VALIDOS
df_validos = df.filter(F.col("tiempo_entrega_min") > 0)

print(f"REGISTROS DESPUES DE FILTRAR: {df_validos.count()}")
print(f"REGISTROS ELIMINADOS: {df.count() - df_validos.count()}")

In [ ]:
# CREAR COLUMNA DERIVADA

df_enriquecido = df_validos.withColumn(
    "tiempo_por_entrega",
    F.col("tiempo_entrega_min") / (F.col("multiples_entregas") + 1)
)

print("Columna 'tiempo_por_entrega' creada")

df_enriquecido.select(
    "id",
    "tiempo_entrega_min",
    "multiples_entregas",
    "tiempo_por_entrega"
).show(5, truncate=False)

In [ ]:
# AGRUPAR POR CIUDAD

resumen_ciudad = df_enriquecido.groupBy("ciudad").agg(
    F.avg("tiempo_entrega_min").alias("tiempo_promedio_entrega"),
    F.min("tiempo_entrega_min").alias("tiempo_minimo"),
    F.max("tiempo_entrega_min").alias("tiempo_maximo"),
    F.count("*").alias("cantidad_pedidos")
).orderBy(F.desc("cantidad_pedidos"))

print("RESUMEN DE ENTREGAS POR CIUDAD")
resumen_ciudad.show(truncate=False)

In [ ]:
# AGRUPAR POR TIPO DE VEHICULO

resumen_vehiculo = df_enriquecido.groupBy("tipo_vehiculo").agg(
    F.avg("tiempo_entrega_min").alias("tiempo_promedio_entrega"),
    F.count("*").alias("cantidad_pedidos")
).orderBy(F.desc("tiempo_promedio_entrega"))

print("RESUMEN POR TIPO DE VEHICULO")
resumen_vehiculo.show(truncate=False)

In [ ]:
# CREAR DIMENSION DE VEHICULOS

vehiculos = [
    ("motorcycle", "Motorizado"),
    ("scooter", "Motorizado"),
    ("electric_scooter", "Motorizado"),
    ("bicycle", "No Motorizado")
]

df_vehiculos = spark.createDataFrame(
    vehiculos,
    ["tipo_vehiculo", "categoria_vehiculo"]
)

print("DIMENSION DE VEHICULOS")
df_vehiculos.show()

In [ ]:
# UNIR LA DIMENSION CON LOS DATOS

df_con_vehiculo = df_enriquecido.join(
    df_vehiculos,
    on="tipo_vehiculo",
    how="left"
)

print(f"Filas antes del JOIN: {df_enriquecido.count()}")
print(f"Filas despues del JOIN: {df_con_vehiculo.count()}")

df_con_vehiculo.select(
    "id",
    "tipo_vehiculo",
    "categoria_vehiculo",
    "tiempo_entrega_min"
).show(5, truncate=False)

In [ ]:
# CREAR VISTA TEMPORAL

df_con_vehiculo.createOrReplaceTempView("entregas_analiticas")

print("Vista temporal 'entregas_analiticas' creada")

In [ ]:
# CONSULTA SQL

consulta = """
SELECT
    categoria_vehiculo,
    COUNT(*) AS cantidad_pedidos,
    ROUND(AVG(tiempo_entrega_min), 2) AS tiempo_promedio,
    ROUND(AVG(tiempo_por_entrega), 2) AS tiempo_promedio_por_entrega
FROM entregas_analiticas
GROUP BY categoria_vehiculo
ORDER BY tiempo_promedio DESC
"""

resultado_sql = spark.sql(consulta)

print("CONSULTA SQL - TIEMPO DE ENTREGA POR CATEGORIA DE VEHICULO")
resultado_sql.show(truncate=False)

In [ ]:
# REVISAR PLAN DE EJECUCION

print("PLAN DE EJECUCION DE LA CONSULTA SQL")
resultado_sql.explain("formatted")

In [ ]:
# REVISAR PARTICIONES

print("PARTICIONES ACTUALES:")
print(df_con_vehiculo.rdd.getNumPartitions())

In [ ]:
# REPARTICIONAR LOS DATOS

df_reparticionado = df_con_vehiculo.repartition(4)

print("PARTICIONES DESPUES DE REPARTICIONAR:")
print(df_reparticionado.rdd.getNumPartitions())

In [ ]:
# GUARDAR DATOS EN PARQUET

ruta_parquet = "/content/datos/analiticos/entregas_parquet"

df_reparticionado.write.mode("overwrite").parquet(ruta_parquet)

print("Datos guardados correctamente en formato Parquet")

In [ ]:
# LEER DATOS DESDE PARQUET

df_parquet = spark.read.parquet(ruta_parquet)

print("DATOS LEIDOS DESDE PARQUET:")
df_parquet.show(5, truncate=False)

print("Cantidad de registros:", df_parquet.count())

In [ ]:
# COMPARACION PANDAS VS SPARK
# COMPARACION PANDAS VS SPARK
df_limpio = pd.read_csv(DATA_PROCESSED / "dataset_limpio.csv")
print("REGISTROS EN PANDAS:", len(df_limpio))
print("REGISTROS EN SPARK:", df_parquet.count())

In [ ]:
# REPORTE DE ARQUITECTURA

reporte = """
ARQUITECTURA BIG DATA - PROYECTO DE LOGISTICA

1. Fuente de datos:
   Dataset limpio en formato CSV.

2. Procesamiento:
   Apache Spark con PySpark.

3. Transformaciones:
   - Filtrado de registros validos.
   - Creacion de tiempo_por_entrega.
   - Agregacion por ciudad.
   - Agregacion por tipo de vehiculo.
   - Union con dimension de vehiculos.
   - Consultas mediante Spark SQL.

4. Particiones:
   - Particiones iniciales: 2
   - Particiones despues de repartition: 4

5. Almacenamiento analitico:
   Formato Parquet.

6. Validacion:
   Pandas y Spark contienen 90000 registros.
"""

print(reporte)

In [ ]:
# DETENER SPARK

spark.stop()

print("Spark detenido correctamente")